# E-Commerce Sales Analytics Dashboard Project

This notebook performs exploratory data analysis on a synthetic Indian e-commerce sales dataset. The analysis is designed for a portfolio-ready analytics dashboard project using Python, SQL, matplotlib, and Power BI dashboard concepts.


## Business Objective

The objective is to understand revenue performance, profit contribution, customer behavior, product performance, payment preferences, and shipping operations for an e-commerce business.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "ecommerce_sales_data.csv"
VISUALS_DIR = PROJECT_ROOT / "visuals"

df = pd.read_csv(DATA_PATH)
df.head()


## Dataset Overview

The dataset includes order-level sales records with customer segment, product category, product name, quantity, price, profit, city, state, payment method, shipping type, and delivery days.


In [ ]:
df.info()
df.describe(include="all")


## 1. Data Cleaning

The cleaning process converts date fields, removes duplicate orders, handles missing values, and creates a monthly period column for trend analysis.


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df = df.drop_duplicates(subset=["order_id"])
df = df.dropna(subset=["order_date", "customer_id", "product_category"])
df["payment_method"] = df["payment_method"].fillna("Unknown")
df["shipping_type"] = df["shipping_type"].fillna("Standard")
df["quantity"] = df["quantity"].fillna(1).astype(int)
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())
df["total_sales"] = df["quantity"] * df["unit_price"]
df["profit"] = df["profit"].fillna(df["profit"].median())
df["month"] = df["order_date"].dt.to_period("M").astype(str)

df.isna().sum()


## 2. KPI Generation

These KPIs are suitable for dashboard cards in Power BI or other BI tools.


In [ ]:
total_revenue = df["total_sales"].sum()
total_profit = df["profit"].sum()
total_orders = df["order_id"].nunique()
average_order_value = total_revenue / total_orders
top_category = df.groupby("product_category")["total_sales"].sum().idxmax()
best_city = df.groupby("city")["total_sales"].sum().idxmax()
most_used_payment = df["payment_method"].mode()[0]

kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue", "Total Profit", "Total Orders", "Average Order Value",
        "Top Category", "Best City", "Most Used Payment Method"
    ],
    "Value": [
        round(total_revenue, 2), round(total_profit, 2), total_orders,
        round(average_order_value, 2), top_category, best_city, most_used_payment
    ]
})
kpis


## 3. Monthly Sales Trend


In [ ]:
monthly_sales = df.groupby("month")["total_sales"].sum()
plt.figure(figsize=(12, 6))
monthly_sales.plot(kind="line", marker="o", color="#0f766e", linewidth=2)
plt.title("Monthly Sales Trend", fontsize=14, weight="bold")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Revenue and Profit by Category


In [ ]:
category_summary = df.groupby("product_category")[["total_sales", "profit"]].sum().sort_values("total_sales", ascending=False)
category_summary


In [ ]:
plt.figure(figsize=(10, 6))
category_summary["total_sales"].sort_values().plot(kind="barh", color="#2f80ed")
plt.title("Revenue by Category", fontsize=14, weight="bold")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
category_summary["profit"].sort_values().plot(kind="barh", color="#27ae60")
plt.title("Profit by Category", fontsize=14, weight="bold")
plt.xlabel("Profit")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


## 5. Customer Segment Analysis


In [ ]:
segment_summary = df.groupby("customer_segment").agg(
    orders=("order_id", "nunique"),
    revenue=("total_sales", "sum"),
    profit=("profit", "sum")
).sort_values("revenue", ascending=False)
segment_summary


In [ ]:
plt.figure(figsize=(8, 6))
df["customer_segment"].value_counts().plot(kind="bar", color="#f2994a")
plt.title("Customer Segment Distribution", fontsize=14, weight="bold")
plt.xlabel("Customer Segment")
plt.ylabel("Order Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. City-Wise Sales Analysis


In [ ]:
city_sales = df.groupby(["city", "state"])["total_sales"].sum().sort_values(ascending=False).head(10)
city_sales


In [ ]:
plt.figure(figsize=(10, 6))
city_sales.sort_values().plot(kind="barh", color="#2f80ed")
plt.title("Top Cities by Sales", fontsize=14, weight="bold")
plt.xlabel("Revenue")
plt.ylabel("City, State")
plt.tight_layout()
plt.show()


## 7. Payment Method Analysis


In [ ]:
payment_summary = df.groupby("payment_method").agg(
    orders=("order_id", "nunique"),
    revenue=("total_sales", "sum")
).sort_values("orders", ascending=False)
payment_summary


In [ ]:
plt.figure(figsize=(8, 6))
df["payment_method"].value_counts().plot(kind="bar", color="#f2994a")
plt.title("Payment Method Usage", fontsize=14, weight="bold")
plt.xlabel("Payment Method")
plt.ylabel("Order Count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 8. Shipping Analysis


In [ ]:
shipping_summary = df.groupby("shipping_type").agg(
    orders=("order_id", "nunique"),
    average_delivery_days=("delivery_days", "mean"),
    revenue=("total_sales", "sum")
).sort_values("orders", ascending=False)
shipping_summary


In [ ]:
plt.figure(figsize=(8, 6))
df["shipping_type"].value_counts().plot(kind="bar", color="#f2994a")
plt.title("Shipping Type Distribution", fontsize=14, weight="bold")
plt.xlabel("Shipping Type")
plt.ylabel("Order Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Top Products Analysis


In [ ]:
top_products = df.groupby("product_name").agg(
    revenue=("total_sales", "sum"),
    units_sold=("quantity", "sum"),
    profit=("profit", "sum")
).sort_values("revenue", ascending=False).head(10)
top_products


In [ ]:
plt.figure(figsize=(10, 6))
top_products["revenue"].sort_values().plot(kind="barh", color="#2f80ed")
plt.title("Top 10 Products by Revenue", fontsize=14, weight="bold")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## Conclusions

- Electronics is the leading revenue category and should remain a priority for campaigns and inventory planning.
- UPI is the most used payment method, showing strong customer preference for quick digital checkout.
- Bengaluru and Hyderabad show strong sales contribution and are good targets for city-specific promotions.
- Consumer customers create the highest order volume, while business segments can be targeted with bulk and loyalty offers.
- Festive and year-end months show stronger sales activity, which supports seasonal campaign planning.


## Business Recommendations

- Increase festive-season promotions for Electronics and Fashion categories.
- Create cross-sell offers for Accessories with Electronics purchases.
- Improve delivery speed in top cities to increase customer satisfaction.
- Use customer segmentation for personalized campaigns.
- Track revenue, profit, AOV, delivery days, and category contribution as recurring dashboard KPIs.
